# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described via a Croissant schema at:
[https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

The dataset provides clinicopathological and molecular characteristics of cancer survivors with a second primary colorectal cancer, including MSI-H status and anatomical distribution. It is organized according to the data standards of MLCommons Croissant.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset's Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the mlcroissant Dataset
dataset = mlc.Dataset(croissant_url)

# Show summary of metadata
meta = dataset.metadata
print(f"Dataset name: {meta.name}")
print(f"Description: {meta.description}\n")
print(f"Identifier: {meta.identifier}")
print(f"Version: {meta.version}")
print(f"License: {meta.license}")
print(f"Published: {meta.datePublished}")
# Optionally, print a few keywords
if hasattr(meta, 'keywords'):
    print(f"Keywords: {', '.join(meta.keywords)}")

## 2. Data Overview
Review available record sets, fields, their `@id`s and the overall data structure.

We'll use only the `@id` for referencing all record sets and fields, as per Croissant best practices.

In [ ]:
# List all available record sets with their @id

record_sets = dataset.metadata.recordSet if hasattr(dataset.metadata, 'recordSet') else []

if not record_sets:
    print('No explicit record sets defined in metadata. Using dataset.columns for tabular extraction.')
    # mlcroissant will often fallback to main dataset table if `recordSet` not populated.
    # Try to extract possible candidate by inspecting the data files info
    if hasattr(dataset.metadata, 'distribution'):
        print('Found distributions in metadata:')
        for dist in dataset.metadata.distribution:
            if hasattr(dist, '@id'):
                print(f"Distribution @id: {dist['@id']}")
    # We'll treat the entire tabular bundle as a single record set.
    # Let's overview the available fields/columns in the dataset.
    print('\nColumns present in the main dataset:')
    for col in dataset.columns:
        # Each column may have @id, name etc.
        col_id = col.get('@id', '<no @id>')
        col_name = col.get('name', '')
        print(f"- Column @id: {col_id}, name: {col_name}")
else:
    print('Record Sets:')
    for rs in record_sets:
        rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else str(rs)
        name = rs.get('name', '<no name>') if isinstance(rs, dict) else ''
        print(f"- RecordSet @id: {rs_id}, name: {name}")
        if 'field' in rs:
            for field in rs['field']:
                field_id = field['@id'] if isinstance(field, dict) and '@id' in field else str(field)
                fname = field.get('name', '<no name>') if isinstance(field, dict) else ''
                print(f"   - Field @id: {field_id}, name: {fname}")

## 3. Data Extraction
Load the tabular data into a DataFrame for analysis.

**Note:** Since the Croissant schema for this dataset does not explicitly define record sets (`recordSet` is empty), we will extract data from the main tabular distribution, referencing columns by their `@id`.

In [ ]:
# Extract all records from the main table, referencing columns by their Croissant @id

data = list(dataset.records())  # returns dicts indexed by @id
df = pd.DataFrame(data)

print(f"Columns (by @id):\n{df.columns.tolist()}")
print(f"\nFirst five records:")
display(df.head())

## 4. Exploratory Data Analysis (EDA)
We will perform several basic operations to get insights:
- Select a numeric field (column) using its `@id`
- Filter records on a numeric threshold
- Normalize the numeric variable
- Group by a categorical field (again using only its `@id`)

Refer to the output above for the exact column `@id` to use.

In [ ]:
# Select a numeric field and a group (categorical) field by their @id

# Let's display available column @ids and names for context
column_ids = [col.get('@id', '') for col in dataset.columns]
column_names = [col.get('name', '') for col in dataset.columns]
columns_info = dict(zip(column_ids, column_names))
print('Available columns for selection:')
for cid, cname in columns_info.items():
    print(f"@id: {cid}, name: {cname}")

# Example selection based on common clinical fields
# Use the actual @id as shown above; here we use placeholder values, replace them as applicable
numeric_field_id = '@age'    # Replace with the actual @id for age, e.g. 'http://mlcommons.org/croissant/fields/age'
group_field_id = '@sex'      # Replace with the actual @id for sex/gender, e.g. 'http://mlcommons.org/croissant/fields/sex'

# Try to auto-select the first numeric and categorical fields if possible
import numpy as np
def infer_numeric_column(df):
    for col in df.columns:
        if np.issubdtype(df[col].dropna().dtype, np.number):
            return col
    return None

def infer_categorical_column(df):
    for col in df.columns:
        if df[col].dtype == 'object' and df[col].nunique() < len(df) // 2:
            return col
    return None

if numeric_field_id not in df.columns:
    inferred_numeric = infer_numeric_column(df)
    print(f"Auto-selected numeric field for EDA: {inferred_numeric}")
    numeric_field_id = inferred_numeric
if group_field_id not in df.columns:
    inferred_categorical = infer_categorical_column(df)
    print(f"Auto-selected grouping field for EDA: {inferred_categorical}")
    group_field_id = inferred_categorical

if numeric_field_id is None:
    raise ValueError("No numeric column found for demonstration.")

# Remove outliers: filter for values above a threshold (example: age > 40)
threshold = 40
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"\nFiltered records where {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalize the numeric column (z-score)
norm_col = numeric_field_id + "_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} (z-score):")
display(filtered_df[[numeric_field_id, norm_col]].head())

# Group by the categorical field (if available), compute mean of numeric
if group_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nMean {numeric_field_id} by {group_field_id} (filtered):")
    display(grouped)
else:
    print(f"Grouping field {group_field_id} is not in DataFrame columns.")

## 5. Visualization
Visualize distributions and group differences using histograms and bar charts.

Replace the example field `@id`s with the actual `@id`s based on your dataset structure if needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(filtered_df[numeric_field_id], bins=15, kde=True)
plt.title(f"Distribution of {numeric_field_id} (filtered >{threshold})")
plt.xlabel(numeric_field_id)
plt.show()

# Visualize group-wise means if grouping field exists
if group_field_id in filtered_df.columns:
    plt.figure(figsize=(6,4))
    sns.barplot(
        data=filtered_df,
        x=group_field_id, y=numeric_field_id, ci=None
    )
    plt.title(f"Mean {numeric_field_id} by group ({group_field_id})")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.show()

## 6. Conclusion
- We've successfully loaded metadata and records from the FAIR^2 dataset using `mlcroissant`.
- The analysis demonstrated how to reference all fields strictly by their Croissant `@id`s.
- We performed basic data filtering, normalization, grouping, and visualization without making any assumptions about column names or positions.

For extended analysis, consult the detailed schema documentation and use the `mlcroissant` introspection tools to explore all fields and metadata.